SmolLM2_135M_MeetNotes_Finetuning.ipynb  (CORRECTED)

Fixes applied vs. the original export — see chat explanation for full rationale:
  1. SFTTrainer/TrainingArguments -> SFTConfig (trl==0.15.2 no longer accepts
     tokenizer / dataset_text_field / max_seq_length / dataset_num_proc / packing
     directly on SFTTrainer — this alone would crash the original script).
  2. tokenizer= -> processing_class= (renamed in trl).
  3. evaluation_strategy -> eval_strategy (deprecated alias).
  4. Added train_on_responses_only so loss is computed on the assistant's
     meeting notes only, not on the system/user prompt (the original notebook's
     own markdown cell told you to do this, then the code never did it).
  5. packing = False, to keep example boundaries intact for response-only
     masking and to stop unrelated meeting transcripts from attending to each
     other inside one packed sequence.
  6. lora_dropout = 0 (Unsloth's fused LoRA kernels are only optimized at 0;
     use dropout only if you actually observe overfitting).
  7. load_in_4bit = False — a 135M model is ~270MB in fp16 and fits a T4
     trivially; 4-bit quantization here only adds quality-degrading noise
     without any real memory benefit.
  8. learning_rate raised to 2e-4 (Unsloth/TRL's own default for LoRA on a
     model this size; 1e-4 is not wrong, just conservative — feel free to
     revert if you want a gentler run).

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1536  # Meeting-notes transcripts are short; raise this only if your transcripts are longer.
dtype = None            # None for auto detection. Float16 for Tesla T4/V100, Bfloat16 for Ampere+.
load_in_4bit = False    # FIX: a 135M model doesn't need 4-bit quantization; keep full precision for quality.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/SmolLM2-135M-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,   # FIX: 0 is required for Unsloth's fast/fused LoRA path. Only raise this
                        # (e.g. to 0.05-0.1) if you empirically see val loss diverging from train loss.
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data Prep

NOTE / thing to verify yourself: this assumes train.jsonl / val.jsonl already contain a
"text" field pre-formatted with the exact ChatML markup below (<|im_start|>system ... etc.),
matching SmolLM2-Instruct's real chat template. If your data-prep pipeline built that string
by hand rather than via `tokenizer.apply_chat_template(...)`, double check it matches exactly
(role tags, newlines, whether a system turn is included) — a mismatch here won't error, it'll
just silently teach the model the wrong prompt format.

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for text in examples["text"]:
        # Appending EOS token is critical to prevent infinite looping during generation
        texts.append(text + EOS_TOKEN)
    return { "text" : texts }

from datasets import load_dataset
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

dataset = dataset.map(formatting_prompts_func, batched=True)

<a name="Train"></a>
### Train the model

FIX: trl==0.15.2's SFTTrainer no longer takes `tokenizer`, `dataset_text_field`,
`max_seq_length`, `dataset_num_proc`, or `packing` directly — they now live on
`SFTConfig`, and the tokenizer is passed as `processing_class`. The original script's
call signature will raise `TypeError: SFTTrainer.__init__() got an unexpected keyword
argument ...` before training even starts.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,          # FIX: was `tokenizer=`
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    args = SFTConfig(                       # FIX: was `TrainingArguments`
        dataset_text_field = "text",        # FIX: moved here from SFTTrainer kwargs
        max_seq_length = max_seq_length,    # FIX: moved here from SFTTrainer kwargs
        dataset_num_proc = 2,               # FIX: moved here from SFTTrainer kwargs
        packing = False,                    # FIX: was True — see rationale below and in chat
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,               # tweak: Unsloth/TRL's usual LoRA default for a model this size
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        eval_strategy = "epoch",            # FIX: was `evaluation_strategy` (deprecated alias)
        save_strategy = "epoch",
        load_best_model_at_end = True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "smollm2_meetnotes_outputs",
        report_to = "none",
    ),
)

# FIX: train on completions only. Without this, the model spends its (very limited,
# 135M-parameter) capacity also learning to predict the system prompt and the user's
# transcript verbatim, instead of concentrating entirely on producing correct meeting
# notes. The original notebook's own markdown cell flagged this TRL feature and then
# never called it. instruction_part / response_part must match your ChatML markup exactly.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference

In [ ]:
FastLanguageModel.for_inference(model)

system_prompt = (
    "You are a meeting notes assistant. Convert the transcript to structured notes. "
    "Use ALL-CAPS headings. Start each bullet with *. No markdown, no extra text."
)
test_transcript = "Okay so let's start the budget meeting. We need to allocate ten thousand dollars for online advertisements next month. Also, John is planning to hire a new QA engineer by August."
user_prompt = f"Transcript:\n{test_transcript}\n\nWrite the meeting notes:"

formatted_prompt = (
    f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
# tweak: skip_special_tokens=True gives you clean output instead of raw <|im_start|>/<|im_end|> tags
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

 Streaming inference 

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_special_tokens = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

<a name="Save"></a>
### Saving, loading finetuned models

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...")
# tokenizer.push_to_hub("your_name/lora_model", token = "...")

# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

# GGUF export (optional — a 135M model is already tiny; q8_0 or f16 GGUF preserves
# quality with negligible size cost, Dynamic 2.0 quants matter most for much larger models)
model.save_pretrained_gguf("smollm2_meetnotes_UD_Q5_K_XL", tokenizer, quantization_method = "UD-Q5_K_XL")